In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [2]:
import logging
import delta_sharing
import pandas as pd
from datetime import datetime

import general_functions.databricks_client as db_client
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import send_to_innkeepr_api_paginated

In [3]:
customer = "to teach"
path_data = f"SprintStories/Project-Intervention-Logging/data/"
url = return_api_url()
print(f"url = {url}")
workspace_id_lists  = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_id_lists if acc["name"] == customer]
try:
    workspace_id = workspace_id[0]
except:
    print("could not be found")
    names = [acc["name"] for acc in workspace_id_lists]
    names = sorted(names)
    print(names)

In [4]:
profile_path = db_client.return_databricks_client()
table_name = "intervention_logging"
table_path = f"{profile_path}#delta_share_events.monitoring.{table_name}"
df = delta_sharing.load_as_pandas(table_path)# , limit=100000)
df.shape

In [5]:
df = df.sort_values(by=["workspace_id","signal_id","created_at"], ascending=False)
df.to_csv(f"{path_data}_intervention_logging.csv", index=False)

In [6]:
unique_signal_ids = df["signal_id"].unique()
print(f"unique signal ids: {unique_signal_ids}")

In [7]:
df.head()

In [8]:
df["manualRetrainReason"].value_counts()

In [9]:
df[df["manualRetrainReason"].isnull()==False]

In [10]:
df[df["actor"]=="human"]

In [11]:
df[df["intervention_id"]=="69b38a25c622c4c20633add5"].values

# Check Model History

In [12]:
workspace_id = df["workspace_id"].unique()
if len(workspace_id) != 1:
    raise ValueError("More than one workspace id found in the table")

workspace_id = workspace_id[0]

# query models
from src.utils.innkeepr_api import send_to_innkeepr_api_paginated
import logging
logger = logging.getLogger(__name__)
models = send_to_innkeepr_api_paginated(f"{url}api/models/query", workspace_id, {}, logger)
models = pd.json_normalize(models)

In [13]:
signal_id = "684549de57bdaeb51e6a04e2"
signal_models = models[models["audience"]==signal_id]
signal_models = signal_models.sort_values(by="created", ascending=False).reset_index(drop=True)
list_models_release = signal_models["id"].tolist()

signal_df = df[df["signal_id"]==signal_id]
signal_df = signal_df.sort_values(by="created_at", ascending=False).reset_index(drop=True)
list_models_release = signal_df["intervention_id"].tolist()

if list_models_release != list_models_release:
    raise ValueError("Models are not the same")


print(f"signals models: {signal_models[['created','id', 'targetingOutlookDays', 'path', 'audienceSizePercentage']]}")
signal_df["changed_fields"] = signal_df["changes"].apply(lambda changes: [x["field"] for x in changes])
signal_df[["created_at", "intervention_id", "prev_intervention_id", "changed_fields", "changes"]]